# 02 — Linear Algebra for ML: PCA, SVD & Eigendecomposition

The three most important matrix factorizations in machine learning:
- **Eigendecomposition**: understanding what a matrix "does"
- **SVD**: the universal factorization — powers recommender systems, NLP, compression
- **PCA from scratch**: dimensionality reduction using eigenvectors

**Prerequisites:** `01_vectors_matrices.ipynb`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_digits

np.random.seed(42)
print('NumPy:', np.__version__)

## 1. Eigenvalues & Eigenvectors

For matrix **A**, vector **v** is an eigenvector if: **Av = λv**

- **v**: direction that A only scales (doesn't rotate)
- **λ**: eigenvalue — how much A scales v

In [ ]:
# Symmetric matrix (real eigenvalues, orthogonal eigenvectors)
A = np.array([[3, 1],
              [1, 3]], dtype=float)

eigenvalues, eigenvectors = np.linalg.eig(A)
print('Eigenvalues λ:', eigenvalues)
print('Eigenvectors V (columns):\n', eigenvectors)

# Verify: Av = λv
for i in range(len(eigenvalues)):
    lam = eigenvalues[i]
    v = eigenvectors[:, i]  # ith column
    Av = A @ v
    lv = lam * v
    print(f'\nλ={lam:.1f}: Av={Av.round(4)}, λv={lv.round(4)}, match={np.allclose(Av, lv)}')

# Eigendecomposition: A = V Λ V^{-1}
V = eigenvectors
Lambda = np.diag(eigenvalues)
A_reconstructed = V @ Lambda @ np.linalg.inv(V)
print('\nReconstruction error:', np.max(np.abs(A - A_reconstructed)))

In [ ]:
# Visualize: eigenvectors as the "natural axes" of the transformation
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: show eigenvectors
ax = axes[0]
theta = np.linspace(0, 2*np.pi, 100)
circle = np.array([np.cos(theta), np.sin(theta)])
transformed = A @ circle

ax.plot(circle[0], circle[1], 'b--', alpha=0.5, label='Unit circle')
ax.plot(transformed[0], transformed[1], 'r-', label='A × unit circle (ellipse)')
for i in range(2):
    v = eigenvectors[:, i] * eigenvalues[i]
    ax.quiver(0, 0, v[0], v[1], scale=1, scale_units='xy', angles='xy', 
              color=['green','purple'][i], width=0.03,
              label=f'Eigenvec {i+1} (λ={eigenvalues[i]:.1f})')
ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)
ax.set_aspect('equal'); ax.legend(fontsize=8)
ax.set_title('Matrix A maps circle → ellipse\nEigenvectors are the ellipse axes')
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
ax.grid(True, alpha=0.3)

# Right: power iteration — largest eigenvalue dominates
ax2 = axes[1]
v = np.random.randn(2)
v /= np.linalg.norm(v)
ratios = []
for _ in range(20):
    v = A @ v
    ratios.append(np.linalg.norm(v))
    v /= np.linalg.norm(v)

ax2.plot(ratios, 'o-', color='steelblue')
ax2.axhline(eigenvalues.max(), color='red', ls='--', label=f'Largest eigenvalue = {eigenvalues.max()}')
ax2.set_xlabel('Iteration'); ax2.set_ylabel('||Av|| before normalization')
ax2.set_title('Power Iteration Converges to Largest Eigenvalue')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Singular Value Decomposition (SVD)

Every matrix **A** (m×n) can be factored as: **A = U Σ V'**

- **U** (m×m): left singular vectors ("output directions")
- **Σ** (m×n): diagonal matrix of singular values ("stretching amounts")
- **V'** (n×n): right singular vectors ("input directions")

SVD generalizes eigendecomposition to non-square matrices.

In [ ]:
# SVD of a matrix
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9],
              [10,11,12]], dtype=float)

U, s, Vt = np.linalg.svd(A, full_matrices=False)
print('A shape:', A.shape)
print('U shape:', U.shape)    # (4,3) — left singular vectors
print('s shape:', s.shape)    # (3,)  — singular values
print('Vt shape:', Vt.shape)  # (3,3) — right singular vectors

print('\nSingular values:', s.round(3))

# Reconstruct: A = U @ diag(s) @ Vt
A_reconstructed = U @ np.diag(s) @ Vt
print('Max reconstruction error:', np.max(np.abs(A - A_reconstructed)).round(10))

In [ ]:
# Low-rank approximation: keep only top k singular values
# This is the basis of image compression and recommender systems!

# Use digits dataset image
digits = load_digits()
img = digits.images[0]  # 8x8 grayscale image

# Use a larger image for demonstration
np.random.seed(42)
img_large = np.random.randn(64, 64)
# Add structure
for i in range(64):
    for j in range(64):
        img_large[i, j] += np.sin(i/5) * np.cos(j/5) * 3

U, s, Vt = np.linalg.svd(img_large, full_matrices=False)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

ks = [1, 2, 5, 64]
for idx, k in enumerate(ks):
    approx = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    err = np.linalg.norm(img_large - approx, 'fro') / np.linalg.norm(img_large, 'fro')
    
    axes[0, idx].imshow(approx, cmap='viridis')
    axes[0, idx].set_title(f'k={k}\nerr={err:.2%}')
    axes[0, idx].axis('off')

# Singular value decay
axes[1, 0].plot(s, 'o-', markersize=4)
axes[1, 0].set_xlabel('Index'); axes[1, 0].set_ylabel('Singular Value')
axes[1, 0].set_title('Singular Value Decay')
axes[1, 0].grid(True, alpha=0.3)

# Explained variance
explained = np.cumsum(s**2) / np.sum(s**2)
axes[1, 1].plot(explained * 100, '-')
axes[1, 1].axhline(95, color='red', ls='--', label='95% threshold')
k_95 = np.argmax(explained >= 0.95) + 1
axes[1, 1].axvline(k_95, color='green', ls='--', label=f'k={k_95}')
axes[1, 1].set_xlabel('k'); axes[1, 1].set_ylabel('% Variance Explained')
axes[1, 1].set_title('Cumulative Variance Explained')
axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

for i in range(2, 4):
    axes[1, i].axis('off')

plt.suptitle('SVD Low-Rank Approximation (Image Compression)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'95% variance captured with k={k_95} out of {len(s)} singular values')

## 3. PCA from Scratch

Principal Component Analysis = SVD on the centered data matrix.

**Steps:**
1. Center data (subtract mean)
2. Compute covariance matrix C = X'X / (n-1)
3. Eigendecompose C (or equivalently, SVD of X)
4. Project onto top k eigenvectors

In [ ]:
class PCAFromScratch:
    """PCA implemented using NumPy SVD."""
    
    def __init__(self, n_components):
        self.n_components = n_components
        self.components_ = None
        self.explained_variance_ratio_ = None
        self.mean_ = None
    
    def fit(self, X):
        # Step 1: Center
        self.mean_ = X.mean(axis=0)
        X_centered = X - self.mean_
        
        # Step 2: SVD of centered data
        U, s, Vt = np.linalg.svd(X_centered, full_matrices=False)
        
        # Step 3: Components are rows of Vt (right singular vectors)
        self.components_ = Vt[:self.n_components]
        
        # Explained variance
        total_var = np.sum(s**2)
        self.explained_variance_ratio_ = (s**2)[:self.n_components] / total_var
        return self
    
    def transform(self, X):
        return (X - self.mean_) @ self.components_.T
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)


# Apply to Iris dataset
iris = load_iris()
X, y = iris.data, iris.target
labels = iris.target_names

pca = PCAFromScratch(n_components=2)
X_pca = pca.fit_transform(X)

print('Original shape:', X.shape)
print('PCA shape:', X_pca.shape)
print('Explained variance ratio:', pca.explained_variance_ratio_.round(3))
print('Total variance explained:', pca.explained_variance_ratio_.sum().round(3))

# Compare with sklearn
from sklearn.decomposition import PCA as SklearnPCA
pca_sk = SklearnPCA(n_components=2).fit_transform(X)
# Signs may differ — compare abs values
print('\nMatches sklearn (up to sign):', np.allclose(np.abs(X_pca), np.abs(pca_sk)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: PCA scatter plot
colors = ['steelblue', 'darkorange', 'green']
ax = axes[0]
for i, label in enumerate(labels):
    mask = y == i
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], 
               c=colors[i], label=label, alpha=0.7, s=50)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('Iris — PCA from Scratch (2D Projection)')
ax.legend(); ax.grid(True, alpha=0.3)

# Right: Scree plot — all 4 components
pca_full = PCAFromScratch(n_components=4).fit(X)
ax2 = axes[1]
ax2.bar(range(1, 5), pca_full.explained_variance_ratio_ * 100, color='steelblue', alpha=0.8)
ax2.plot(range(1, 5), np.cumsum(pca_full.explained_variance_ratio_) * 100, 
         'ro-', lw=2, label='Cumulative')
ax2.axhline(95, color='red', ls='--', alpha=0.5, label='95% threshold')
ax2.set_xlabel('Principal Component'); ax2.set_ylabel('Variance Explained (%)')
ax2.set_title('Scree Plot — Iris Dataset')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Matrix Rank & Null Space

In [ ]:
# Rank: number of linearly independent rows/columns
A_full = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]])  # rank 3
A_rank2 = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])  # rank 2 (row3 = row1 + row2)
A_rank1 = np.array([[1, 2, 3], [2, 4, 6], [3, 6, 9]])  # rank 1

print('rank(A_full):', np.linalg.matrix_rank(A_full))
print('rank(A_rank2):', np.linalg.matrix_rank(A_rank2))
print('rank(A_rank1):', np.linalg.matrix_rank(A_rank1))

# Connection to SVD: rank = number of non-zero singular values
for name, M in [('A_full', A_full), ('A_rank2', A_rank2), ('A_rank1', A_rank1)]:
    s = np.linalg.svd(M, compute_uv=False)
    print(f'{name} singular values: {s.round(2)}')

print('\n--- ML Implication ---')
print('A matrix with rank < min(m,n) is "rank deficient".')
print('In regression: X is rank deficient when features are perfectly collinear.')
print('Solution: Regularization (Ridge adds λI to X\'X, making it full rank).')

## 5. Applications in ML

| Technique | Linear Algebra | Use |
|-----------|---------------|-----|
| PCA | SVD / Eigendecomposition | Dimensionality reduction, visualization |
| LSA / LSI | SVD | Document embeddings, topic modeling |
| Recommender SVD | SVD | Matrix factorization for ratings |
| Linear Regression | Least squares / pseudoinverse | Closed-form solution |
| Neural network forward pass | Matrix multiply | Every layer |
| Ridge Regression | (X'X + λI)⁻¹X'y | Regularized regression |
| Attention mechanism | QK'V / sqrt(d) | Transformers |

**Next:** [03_calculus_gradient_descent.ipynb](03_calculus_gradient_descent.ipynb)